# Part 5 — one clock, honest captions

Run all cells in VS Code Jupyter using the project environment. No narration recording
was supplied: the demonstration timing is **estimated**, not verified speech alignment.
Authoring/export cells require only the library; the final two render-dependent cells
report explicit skips without Manim. CI uses strict mode to disallow those skips.

In [ ]:
from pathlib import Path
import sys, os, importlib.util
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Open the repository workspace first"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
from examples.subtitle_sync_demo import build_timeline, export_demo
from quonitron_manim.timeline import validate_timeline, compile_schedule
from quonitron_manim.timeline.report import format_timeline_report
from IPython.display import display, Markdown, Video
timeline = build_timeline()
assert all(c.timing_status == "estimated" for c in timeline.cues)
print(timeline.narration)

In [ ]:
report = validate_timeline(timeline)
report.raise_if_invalid()
display(Markdown(format_timeline_report(timeline)))
print("Shared markers:", [(m.time, m.kind, m.ref_id) for m in compile_schedule(timeline)])
assert not timeline.cues_at(3.7), "Intentional gap"
assert timeline.cues_at(4)[0].id == "camera-explanation"

In [ ]:
from quonitron_manim.timeline import Timeline
aligned = Timeline.from_dict(timeline.to_dict())
aligned.cues[0].align(.05, 3.55, notes="Example editorial adjustment, not audio review")
assert aligned.cues[0].timing_status == "manually_aligned"
assert timeline.cues[0].timing_status == "estimated"  # original unchanged
out = ROOT / "media/part5/notebook"
exports = export_demo(out)
print(exports["srt_text"])
print(exports["vtt_text"])
assert exports["track"].cues[0].timing_status == "estimated"

In [ ]:
can_render = importlib.util.find_spec("manim") is not None
if os.environ.get("QUONITRON_REQUIRE_RENDER") and not can_render:
    raise RuntimeError("Strict notebook requires native Manim/Cairo/Pango")
if can_render:
    from quonitron_manim.components.foundation.subtitle_overlay import SubtitleOverlay
    overlay = SubtitleOverlay()
    print("Measured active combinations:", overlay.preflight(timeline))
    from manim import tempconfig
    from examples.subtitle_sync_demo import SubtitleSyncDemo
    with tempconfig({"quality": "low_quality", "media_dir": str(out),
                     "disable_caching": True, "write_to_movie": True,
                     "preview": False, "verbosity": "ERROR"}):
        scene = SubtitleSyncDemo()
        scene.render()
        collisions = scene.check_layout()
        assert not [i for i in collisions if i.severity == "ERROR"]
        movie = Path(scene.renderer.file_writer.movie_file_path)
        assert movie.stat().st_size > 0
        print("Actual render:", movie, "Final geometry:", collisions)
else:
    print("RENDER SKIPPED: Manim is absent; measured Pango/collision/render checks not executed.")

In [ ]:
if can_render:
    display(Video(str(movie), embed=True))
    print("HTML embedded video generated. Watch transitions and captions in VS Code; backend execution is not frontend playback signoff.")
else:
    print("RENDER SKIPPED: inline preview unavailable without a rendered movie.")

## Manual review checklist
Inspect the first cue, the 3.6–3.8 second gap, caption continuity across the 4 second
beat/camera transition, and the two-line recap after 7.2 seconds. Check punctuation,
font size, safe margins, and important-content collisions. Listen and align against
an actual recording before declaring `verified`, including reviewer notes and an
`audio_reference`. Sampled geometry is not visual approval or forced alignment.